In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Шаг 1: Определите модели Pydantic для структурированных ответов

Эти модели определяют **схему**, которую агенты будут возвращать. Использование `response_format` с Pydantic обеспечивает:
- ✅ Типобезопасное извлечение данных
- ✅ Автоматическую валидацию
- ✅ Отсутствие ошибок разбора из-за свободного текста в ответах
- ✅ Простую условную маршрутизацию на основе полей


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Шаг 2: Создание инструмента бронирования отеля

Этот инструмент будет вызываться **availability_agent** для проверки доступности номеров. Мы используем декоратор `@ai_function`, чтобы:
- Преобразовать функцию Python в вызываемый ИИ инструмент
- Автоматически сгенерировать JSON-схему для LLM
- Обрабатывать проверку параметров
- Включить автоматический вызов агентами

Для этого демо:
- **Стокгольм, Сиэтл, Токио, Лондон, Амстердам** → Номера есть ✅
- **Все остальные города** → Номеров нет ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Шаг 3: Определение функций условий для маршрутизации

Эти функции анализируют ответ агента и определяют, какой путь выбрать в рабочем процессе.

**Основной шаблон:**
1. Проверить, является ли сообщение `AgentExecutorResponse`
2. Распарсить структурированный вывод (модель Pydantic)
3. Вернуть `True` или `False` для управления маршрутизацией

Рабочий процесс будет оценивать эти условия на **ветвях**, чтобы решить, какой исполнитель вызывать дальше.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Шаг 4: Создание пользовательского дисплейного исполнителя

Исполнители — это компоненты рабочего процесса, которые выполняют преобразования или побочные эффекты. Мы используем декоратор `@executor` для создания пользовательского исполнителя, который отображает окончательный результат.

**Ключевые понятия:**
- `@executor(id="...")` — регистрирует функцию как исполнителя рабочего процесса
- `WorkflowContext[Never, str]` — подсказки типов для входных и выходных данных
- `ctx.yield_output(...)` — возвращает окончательный результат рабочего процесса


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Шаг 5: Загрузка переменных окружения

Настройте клиент LLM. Этот пример работает с:
- **Моделями GitHub** (бесплатный уровень с токеном GitHub)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шаг 6: Создание AI Агентов со Структурированными Выходными Данными

Мы создаем **три специализированных агента**, каждый из которых обернут в `AgentExecutor`:

1. **availability_agent** - Проверяет доступность отелей с помощью инструмента
2. **alternative_agent** - Предлагает альтернативные города (когда нет свободных номеров)
3. **booking_agent** - Побуждает к бронированию (когда номера доступны)

**Ключевые особенности:**
- `tools=[hotel_booking]` - Предоставляет агенту инструмент
- `response_format=PydanticModel` - Обеспечивает структурированный вывод в формате JSON
- `AgentExecutor(..., id="...")` - Оборачивает агента для использования в рабочем процессе


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Шаг 7: Создание рабочего процесса с условными ребрами

Теперь мы используем `WorkflowBuilder` для построения графа с условной маршрутизацией:

**Структура рабочего процесса:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Ключевые методы:**
- `.set_start_executor(...)` - Устанавливает точку входа
- `.add_edge(from, to, condition=...)` - Добавляет условное ребро
- `.build()` - Завершает создание рабочего процесса


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: Запуск теста 1 - Город БЕЗ доступности (Париж)

Давайте проверим путь **без доступности**, запросив отели в Париже (в нашей симуляции там нет свободных номеров).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Шаг 9: Запустите тестовый пример 2 - Город С наличием (Стокгольм)

Теперь давайте протестируем путь **наличия**, запросив отели в Стокгольме (в котором есть номера в нашей симуляции).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Основные выводы и дальнейшие шаги

### ✅ Что вы узнали:

1. **Паттерн WorkflowBuilder**
   - Используйте `.set_start_executor()` для определения точки входа
   - Используйте `.add_edge(from, to, condition=...)` для условной маршрутизации
   - Вызовите `.build()`, чтобы завершить построение рабочего процесса

2. **Условная маршрутизация**
   - Функции условий анализируют `AgentExecutorResponse`
   - Разбирают структурированные выходные данные для принятия решений о маршруте
   - Возвращают `True`, чтобы активировать ребро, `False` — чтобы пропустить его

3. **Интеграция инструментов**
   - Используйте `@ai_function` для преобразования функций Python в инструменты ИИ
   - Агенты автоматически вызывают инструменты при необходимости
   - Инструменты возвращают JSON, который агенты могут парсить

4. **Структурированные выходные данные**
   - Используйте модели Pydantic для типобезопасного извлечения данных
   - Устанавливайте `response_format=MyModel` при создании агентов
   - Разбирайте ответы с помощью `Model.model_validate_json()`

5. **Пользовательские исполнители**
   - Используйте `@executor(id="...")` для создания компонентов рабочего процесса
   - Исполнители могут трансформировать данные или выполнять побочные эффекты
   - Используйте `ctx.yield_output()`, чтобы выдавать результаты рабочего процесса

### 🚀 Применение на практике:

- **Бронирование путешествий**: Проверка доступности, предложение альтернатив, сравнение вариантов
- **Обслуживание клиентов**: Маршрутизация по типу проблемы, настроению, приоритету
- **Электронная коммерция**: Проверка наличия, предложение альтернатив, обработка заказов
- **Модерация контента**: Маршрутизация по оценкам токсичности, сигналам пользователей
- **Процессы утверждения**: Маршрутизация по сумме, роли пользователя, уровню риска
- **Многоступенчатая обработка**: Маршрутизация по качеству данных, полноте

### 📚 Следующие шаги:

- Добавить более сложные условия (несколько критериев)
- Реализовать циклы с управлением состоянием рабочего процесса
- Добавить подрабочие процессы для переиспользуемых компонентов
- Интегрировать с реальными API (бронирование отелей, системы инвентаризации)
- Добавить обработку ошибок и запасные пути
- Визуализировать рабочие процессы с помощью встроенных инструментов визуализации


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
